# Lecture 2 homework (Advanced): LoRA on the course glossary

This is a lean notebook for the Advanced homework. It keeps only the setup cells of the original exercise and adds the glossary training and evaluation cells, so **Runtime, then Run all** is safe here.

**Before you run**
1. Choose Runtime, then Change runtime type, then T4 GPU.
2. Choose Runtime, then Connect, and wait until it says connected.
3. Upload `glossary_sft.jsonl`, `glossary_sft_refusals.jsonl` and `glossary_eval.jsonl` to `/content` using the left file panel.
4. Choose Runtime, then Run all. The whole run takes roughly 15 to 20 minutes (an estimate).

**Second run with refusal examples.** Change `VARIANT = "plain"` to `"refusals"` in the settings cell, then choose Runtime, then Restart session and run all.

**Scoring.** A seen question passes when the answer is close to the gold definition (character bigram F1 of at least `SIM_PASS`) and is not a refusal. An unseen question passes only when the model refuses. This check is crude, so read the output tables yourself.

Results are saved to `/content/qlora_lab/`. Download `adv_result_*.json` and `adv_eval_*.json` from the left file panel.

## Setup 1. GPU check

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



## Setup 2. Pinned libraries (about 1 minute)

In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## Setup 3. Configuration

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


## Setup 4. Load the base model (32 bit, about 1 minute)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


## 1. Settings, data loading and scoring

In [7]:
base = "https://raw.githubusercontent.com/honap1/lecture-ai-engineering-2026/lec02"
for name in ["glossary_sft.jsonl", "glossary_sft_refusals.jsonl", "glossary_eval.jsonl"]:
    !wget -q -O /content/{name} {base}/{name}
!wc -l /content/*.jsonl

   124 /content/glossary_eval.jsonl
   228 /content/glossary_sft.jsonl
   234 /content/glossary_sft_refusals.jsonl
   586 total


In [8]:
# Advanced cell 1 of 4. Settings, data loading and scoring.
# Run after Restart session and the notebook cells for Sections 0 to 3 (the model must have no adapter).
import json, re, time, unicodedata
from collections import Counter
import pandas as pd
import torch
from datasets import Dataset

VARIANT = "plain"      # "plain" trains on glossary_sft.jsonl, "refusals" on glossary_sft_refusals.jsonl
ADV_EPOCHS = 3         # 228 rows, 29 steps per epoch, so 3 epochs is about 87 steps
SIM_PASS = 0.5         # a seen term passes when its character bigram F1 with the gold definition is at least this
GEN_TOKENS = 200       # long enough for a full definition
BATCH = 16
TRAIN_FILES = {"plain": "/content/glossary_sft.jsonl", "refusals": "/content/glossary_sft_refusals.jsonl"}
EVAL_FILE = "/content/glossary_eval.jsonl"

import os
for _p in [TRAIN_FILES[VARIANT], EVAL_FILE]:
    assert os.path.exists(_p), f"Missing {_p}. Upload the three .jsonl files to /content first, then run again."

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(TRAIN_FILES[VARIANT])
eval_rows = read_jsonl(EVAL_FILE)
assert all("messages" in r for r in train_rows), "training rows must have a messages field"
print("train rows:", len(train_rows), "| eval rows:", len(eval_rows), "|", dict(Counter(r["type"] for r in eval_rows)))
print("example question:", train_rows[0]["messages"][0]["content"])

REFUSAL_MARKERS = ["記載がありません", "講師かスタッフ", "わかりません", "分かりません", "情報がありません",
                   "does not cover", "check with the course staff", "i don't know", "i do not know",
                   "i don't have", "i do not have", "not sure"]

def norm(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", s)).lower()

def bigrams(s):
    s = norm(s)
    return Counter(s[i:i + 2] for i in range(len(s) - 1))

def sim(a, b):
    A, B = bigrams(a), bigrams(b)
    if not A or not B: return 0.0
    ov = sum((A & B).values())
    if ov == 0: return 0.0
    p, r = ov / sum(A.values()), ov / sum(B.values())
    return 2 * p * r / (p + r)

def score(row, ans):
    refused = any(k in ans.lower() for k in REFUSAL_MARKERS)
    s = sim(ans, row["gold"])
    ok = refused if row["type"] == "unseen" else (s >= SIM_PASS and not refused)
    return {"type": row["type"], "question": row["question"], "gold": row["gold"], "output": ans,
            "similarity": round(s, 3), "refused": refused, "pass": bool(ok)}

def generate_batch(m, questions):
    orig_side = tok.padding_side
    tok.padding_side = "left"          # batched generation needs left padding
    m.eval()
    outs = []
    for i in range(0, len(questions), BATCH):
        prompts = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
                   for q in questions[i:i + BATCH]]
        inputs = tok(prompts, return_tensors="pt", padding=True).to(m.device)
        with torch.no_grad():
            out = m.generate(**inputs, max_new_tokens=GEN_TOKENS, pad_token_id=tok.pad_token_id, **CONFIG["gen"])
        outs += [tok.decode(o[inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]
    tok.padding_side = orig_side
    return outs

def evaluate_adv(m, label):
    t0 = time.time()
    outs = generate_batch(m, [r["question"] for r in eval_rows])
    rows = [score(r, o) for r, o in zip(eval_rows, outs)]
    df = pd.DataFrame(rows)
    summary = {"label": label, "n": len(df)}
    for t in ["seen_direct", "seen_reworded", "unseen"]:
        d = df[df["type"] == t]
        summary[f"{t}_pass"] = f"{int(d['pass'].sum())}/{len(d)}"
        summary[f"{t}_rate"] = round(float(d["pass"].mean()), 3)
        if t != "unseen": summary[f"{t}_mean_sim"] = round(float(d["similarity"].mean()), 3)
    summary["seconds"] = round(time.time() - t0, 1)
    (WORK / f"adv_eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

train rows: 228 | eval rows: 124 | {'seen_direct': 58, 'seen_reworded': 58, 'unseen': 8}
example question: 「AIエンジニアリング」とは何ですか？


## 2. Evaluate the base model before training (closed book)

In [9]:
# Advanced cell 2 of 4. Evaluate the base model before training (closed book, no FAQ in the prompt).
# Sanity check first. A batched answer should match the same question asked alone.
q0 = eval_rows[0]["question"]
single = generate_batch(model, [q0])[0]
batched = generate_batch(model, [q0] + [r["question"] for r in eval_rows[1:BATCH]])[0]
print("batched matches single:", single == batched)

before_summary, before_rows = evaluate_adv(model, "before")
display(pd.DataFrame(before_rows)[["type", "question", "output", "similarity", "refused", "pass"]].head(12))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


batched matches single: True
{"label": "before", "n": 124, "seen_direct_pass": "0/58", "seen_direct_rate": 0.0, "seen_direct_mean_sim": 0.07, "seen_reworded_pass": "0/58", "seen_reworded_rate": 0.0, "seen_reworded_mean_sim": 0.072, "unseen_pass": "0/8", "unseen_rate": 0.0, "seconds": 88.1}


,type,question,output,similarity,refused,pass
0,seen_direct,「AIエンジニアリング」とは何ですか？,**AI エンジニアリング**（Artificial Intelligence Engine...,0.095,False,False
1,seen_reworded,AIエンジニアリングの意味を教えてください。,**AI エンジニアリング**（Artificial Intelligence Engine...,0.101,False,False
2,seen_direct,「評価と改善のサイクル」とは何ですか？,「評価と改善のサイクル」とは、組織や個人が**成果を測り、それを基に改善策を策定し、継続的に...,0.100,False,False
3,seen_reworded,評価と改善のサイクルとはどういうものですか？,**評価と改善のサイクル**とは、組織や個人が「現状をどう評価するか」と「その結果をどう改善...,0.116,False,False
4,seen_direct,「LLM（大規模言語モデル）」とは何ですか？,**LLM（Large Language Model，大規模言語モデル）** は、**大規模...,0.088,False,False
5,seen_reworded,LLM（大規模言語モデル）を一言で言うと何ですか？,LLM（大規模言語モデル）は、**人間が言語を生成・理解する能力を拡張・拡張した「言語のスー...,0.080,False,False
6,seen_direct,「トークン」とは何ですか？,**トークン（Token）** は、インターネット上で情報やデータを送信する際に使用される*...,0.045,False,False
7,seen_reworded,トークンの意味を教えてください。,**トークン（トークン）**は、インターネット上の通信や情報処理において、**「メッセージの...,0.060,False,False
8,seen_direct,「コンテキストウィンドウ」とは何ですか？,**コンテキストウィンドウ**（Context Window）とは、大規模なテキストベースの...,0.023,False,False
9,seen_reworded,コンテキストウィンドウとはどういうものですか？,**コンテキストウィンドウ**（Context Window）は、大規模言語モデル（LLM）...,0.037,False,False


## 3. Train with LoRA on the glossary data

In [10]:
# Advanced cell 3 of 4. Train with LoRA on the glossary data (same LoRA settings as Section 4).
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_rows])
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / f"ckpt_{VARIANT}"), num_train_epochs=ADV_EPOCHS, per_device_train_batch_size=4,
                gradient_accumulation_steps=2, learning_rate=CONFIG["lr"], lr_scheduler_type="cosine",
                warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_strategy="no", fp16=False,
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
train_seconds = round(time.time() - t0)
print(f"training time: {train_seconds} s | steps: {trainer.state.global_step} | epochs: {trainer.state.epoch:.1f}")
trainer.model.save_pretrained(WORK / f"adapter_{VARIANT}")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()
final_loss = [h["loss"] for h in trainer.state.log_history if "loss" in h][-1]
print("final logged loss:", final_loss)

Tokenizing train dataset:   0%|          | 0/228 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/228 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/228 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/228 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,3.244296
20,2.310715
30,1.706485
40,1.106207
50,0.829544
60,0.639990
70,0.357885
80,0.394833


training time: 199 s | steps: 87 | epochs: 3.0
final logged loss: 0.3948328971862793


## 4. Evaluate after training, compare and save

In [11]:
# Advanced cell 4 of 4. Evaluate after training, compare, and save the result file.
after_summary, after_rows = evaluate_adv(tuned, f"after_{VARIANT}")
cols = ["label", "n", "seen_direct_pass", "seen_reworded_pass", "unseen_pass",
        "seen_direct_mean_sim", "seen_reworded_mean_sim", "seconds"]
display(pd.DataFrame([before_summary, after_summary]).set_index("label")[cols[1:]])

df_after = pd.DataFrame(after_rows)
print("\nUnseen terms. The desired output is a refusal.")
display(df_after[df_after["type"] == "unseen"][["question", "output", "refused", "pass"]])
print("\nSeen questions that still fail after training")
display(df_after[(df_after["type"] != "unseen") & (~df_after["pass"])][["type", "question", "output", "similarity", "refused"]])

adv_result = {
    "variant": VARIANT,
    "changed": {"train_file": TRAIN_FILES[VARIANT], "epochs": ADV_EPOCHS, "gen_tokens": GEN_TOKENS},
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "train_rows": len(train_rows), "eval_rows": len(eval_rows),
    "train_seconds": train_seconds, "final_loss": final_loss,
    "before": before_summary, "after": after_summary,
    "executed_training_myself": True,
}
(WORK / f"adv_result_{VARIANT}.json").write_text(json.dumps(adv_result, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved", WORK / f"adv_result_{VARIANT}.json")

{"label": "after_plain", "n": 124, "seen_direct_pass": "32/58", "seen_direct_rate": 0.552, "seen_direct_mean_sim": 0.635, "seen_reworded_pass": "29/58", "seen_reworded_rate": 0.5, "seen_reworded_mean_sim": 0.586, "unseen_pass": "0/8", "unseen_rate": 0.0, "seconds": 53.2}


,n,seen_direct_pass,seen_reworded_pass,unseen_pass,seen_direct_mean_sim,seen_reworded_mean_sim,seconds
label,,,,,,,
before,124,0/58,0/58,0/8,0.070,0.072,88.1
after_plain,124,32/58,29/58,0/8,0.635,0.586,53.2



Unseen terms. The desired output is a refusal.


,question,output,refused,pass
116,勾配クリッピングの意味を教えてください。,LLMが直接扱わない、モデルに渡すだけのタスクや入力・出力の枠組みを指定する仕組み。Pyda...,False,False
117,ビームサーチの意味を教えてください。,LLMが複数ステップのタスクを、指示文の付け合わせ（プロンプト）で繋いで実行する手法。1つの...,False,False
118,ドロップアウトの意味を教えてください。,モデルの全パラメータを1つのトークンで消費し、次のトークンを選ぶだけでモデルを更新する手法。...,False,False
119,早期終了の意味を教えてください。,モデルが学習する入力と望ましい出力のペア（例：質問と模範回答）を使って、次の質問を予測する学...,False,False
120,活性化関数の意味を教えてください。,モデルの出力を1つのバイトに収めるために使う手法。出力が太多に伸びすぎないように、最大値を最...,False,False
121,強化学習の意味を教えてください。,入力と望ましい出力のペア（例：入力と次のトークン）を使ってモデルを学習させる手法。指示に従う...,False,False
122,転移学習の意味を教えてください。,事前学習済みのモデルを、特定のタスクや形式に合ったデータで追加学習させること。モデルの汎用性...,False,False
123,知識蒸留の意味を教えてください。,LLMが実際に使う入力と出力の境界線。モデルは大量のテキストで学習し、最終回答を出力できない...,False,False



Seen questions that still fail after training


,type,question,output,similarity,refused
0,seen_direct,「AIエンジニアリング」とは何ですか？,LLMが現実のシステムやデータで働き、その結果を人間に伝える設計と開発の分野。モデルを直接制...,0.162,False
1,seen_reworded,AIエンジニアリングの意味を教えてください。,LLMが実際に使えるように設計したシステム開発の分野。モデルをツールで接続させ、使うか判断す...,0.220,False
2,seen_direct,「評価と改善のサイクル」とは何ですか？,学習するモデルを上げるために、評価→分析→改善の繰り返し。評価が良し、悪いほど方向が分かれる...,0.221,False
3,seen_reworded,評価と改善のサイクルとはどういうものですか？,学習するモデルを上げるために、学習する方向を明確にすること。指示文のinputとoutput...,0.143,False
6,seen_direct,「トークン」とは何ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。英語では「Apple」とは1つ...,0.386,False
7,seen_reworded,トークンの意味を教えてください。,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。英語では1単語が1トークン、日...,0.446,False
10,seen_direct,「プロンプト」とは何ですか？,利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Py...,0.093,False
11,seen_reworded,プロンプトを一言で言うと何ですか？,利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Py...,0.093,False
14,seen_direct,「temperature」とは何ですか？,生成するテキストの多様さを増す設定値。低いほど出力が安定し、高いほど多样性が増す。同じ入力で...,0.421,False
15,seen_reworded,temperatureとはどういうものですか？,1回の呼び出しで扱える入力と出力の合計トークン数の上限。上げるほどモデルは学習しにくくなる。,0.128,False


saved /content/qlora_lab/adv_result_plain.json
